# The Coherent Coupler system of Jensen


In this notebook we explore how the general case we consider relates to Jensen's coherent coupler model. We show that Jensen's is a special case in which the quartic term can be removed by a linear transformation that diagonalises the wave mixing linear coupling. We show that in the general case such a transform creates a complicated mess of new wave mixing terms that does not deliver the same simplicity and is thus not a practical path for obtaining solutions and analysing systems.

In [75]:
from sympy import *
from time import time

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi) = symbols(
    '''delta, nu, Aeff, chi'''
)

gbar2 = Symbol('gbar2', latex_name=r'\bar{g_2}')
gbar3 = Symbol('gbar3', latex_name=r'\bar{g_3}')
gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

Det = Function('Det') # Unevaluated determinant

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')
rhohat = Function('rhohat', latex_name=r'\hat{rho}')

phi = Function('phi')
varphi = Function('varphi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

A = Function('A')
Ac = Function('Ac')
B = Function('B')
Bc = Function('Bc')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

kappa = IndexedBase('kappa')
Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammahat = IndexedBase('gammahat', latex_name=r'\hat{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lamda = IndexedBase('lamda') # special sympy spelling
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
ajk_syms = [(a[2, 1], a[1, 2])]

uv_j_rho = Eq(u(z,mu[j])*v(z,mu[j]), gamma[j] - rho(z))

duvj = Eq(
    Derivative(u(z, mu[j])*v(z, mu[j]),z), 
    Product(v(z, mu[k]), (k, 1, 2)) - Product(u(z, mu[k]), (k, 1, 2))
)

uprodj = Eq(
    Product(u(z, mu[j]), (j, 1, 2)),
    -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + 
    Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + 
    Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2
)
vprodj = Eq(uprodj.lhs + duvj.rhs.replace(k,j), uprodj.rhs + duvj.lhs.replace(k,j))
a_0_eq = Eq(uprodj.rhs + vprodj.rhs, uprodj.lhs + vprodj.lhs)

du_phase_mod_part = (a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*u(z,mu[j])
dv_phase_mod_part = -(a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*v(z,mu[j])

du_mixing_part = Product(v(z,mu[k]), (k,1,2))/v(z, mu[j])
dv_mixing_part = -Product(u(z,mu[k]), (k,1,2))/u(z, mu[j])

duj = Eq(diff(u(z,mu[j]),z), -du_phase_mod_part + du_mixing_part)
dvj = Eq(diff(v(z,mu[j]),z), (-dv_phase_mod_part).factor() + dv_mixing_part)
duj_subs = [duj.subs(j, _j + 1).args for _j in range(2)]
dvj_subs = [dvj.subs(j, _j + 1).args for _j in range(2)]
duj_dvj_subs = duj_subs + dvj_subs


assert (
    diff(solve(a_0_eq.doit(),a[0])[0],z).subs(duj_dvj_subs).doit().subs(ajk_syms).simplify() == 0
), 'a0 not conserved'

uv_j_rho

uprodj
vprodj
a_0_eq

duj
dvj
duvj

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(Product(u(z, mu[j]), (j, 1, 2)), -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(Product(v(z, mu[j]), (j, 1, 2)), Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(a[0] + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)), Product(u(z, mu[j]), (j, 1, 2)) + Product(v(z, mu[j]), (j, 1, 2)))

Eq(Derivative(u(z, mu[j]), z), -(a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*u(z, mu[j]) + Product(v(z, mu[k]), (k, 1, 2))/v(z, mu[j]))

Eq(Derivative(v(z, mu[j]), z), (a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*v(z, mu[j]) - Product(u(z, mu[k]), (k, 1, 2))/u(z, mu[j]))

Eq(Derivative(u(z, mu[j])*v(z, mu[j]), z), -Product(u(z, mu[k]), (k, 1, 2)) + Product(v(z, mu[k]), (k, 1, 2)))

In [13]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]

g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _

g2_dj
g3_dj

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2])

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)

Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

In [28]:
Lams = [
    Eq(
        Lamda[0, m],
        -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
        + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
    ),
    Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))
]

sum_r1j_1 = Eq(Sum(r[1, j], (j, 1, 2)), 1)
r1j_log_part = Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))
r2j_log_part = Eq(r[2, j], Lamda[1, j]/b[2] - Rational(1,2))

In [46]:
uz_sol_no_sqrt = Eq(
    u(z, mu[j]),
    sqrt(wpp(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)*
    exp(
        z*r[0, j] + 
        log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j]
        + epsilon[j]
    )/(
        sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*
        sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 - z1, g2, g3)*sqrt(b[2])
    )
)
vz_sol_no_sqrt = Eq(
    v(z, mu[j]),
    sqrt(wpp(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[j], g2, g3)*
    exp(
        -z*r[0, j] - 
        log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] 
        - epsilon[j]
    )/(
        sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*
        sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 + z1, g2, g3)*sqrt(b[2])
    )
)


uz_sol_no_sqrt
vz_sol_no_sqrt

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 - z1, g2, g3)*sqrt(b[2])))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 + z1, g2, g3)*sqrt(b[2])))

## The special case of Jensen and the effect of linear diagonlisation of wave mixing

In this section we show how the abstract coherent coupler model we have been considering is related to the physical model considered by Jensen. The Jensen case is in fact a very special configuration of parameters for which $b_1=0$ and when the wave mixing terms are diagonalised becomes a polarisation like model that contains no quartic.

In [16]:
_vu2_switch = [(v(z, mu[2]),x), (u(z, mu[2]), v(z, mu[2])), (x, u(z, mu[2]))]
a_0_eq_swtch = a_0_eq.doit().subs(_vu2_switch)
H_swtch = Eq(
    a_0_eq_swtch.rhs - a_0_eq_swtch.rhs + a[0], 
    (a_0_eq_swtch.rhs - a_0_eq_swtch.lhs + a[0])
    .subs(ajk_syms)
)

du_dv_swtch = [
    Eq(diff(u(z, mu[1]),z), diff(H_swtch.rhs, v(z, mu[1])).simplify().collect([u(z, mu[1]), u(z, mu[2])])),
    Eq(diff(u(z, mu[2]),z), diff(H_swtch.rhs, v(z, mu[2])).simplify().collect([u(z, mu[1]), u(z, mu[2])])),
    Eq(diff(v(z, mu[1]),z), -diff(H_swtch.rhs, u(z, mu[1])).simplify().collect([v(z, mu[1]), v(z, mu[2])])),
    Eq(diff(v(z, mu[2]),z), -diff(H_swtch.rhs, u(z, mu[2])).simplify().collect([v(z, mu[1]), v(z, mu[2])]))   
]
du_dv_swtch_subs = [_.args for _ in du_dv_swtch]

for _ in du_dv_swtch:
    _
    
H_swtch

Eq(Derivative(u(z, mu[1]), z), (-u(z, mu[2])*v(z, mu[2])*a[1, 2] - a[1])*u(z, mu[1]) - u(z, mu[1])**2*v(z, mu[1])*a[1, 1] + u(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), (-u(z, mu[2])*v(z, mu[1])*a[1, 2] + 1)*u(z, mu[1]) - u(z, mu[2])**2*v(z, mu[2])*a[2, 2] - u(z, mu[2])*a[2])

Eq(Derivative(v(z, mu[1]), z), -(-u(z, mu[2])*v(z, mu[2])*a[1, 2] - a[1])*v(z, mu[1]) + u(z, mu[1])*v(z, mu[1])**2*a[1, 1] - v(z, mu[2]))

Eq(Derivative(v(z, mu[2]), z), -(-u(z, mu[1])*v(z, mu[2])*a[1, 2] + 1)*v(z, mu[1]) + u(z, mu[2])*v(z, mu[2])**2*a[2, 2] + v(z, mu[2])*a[2])

Eq(a[0], -(u(z, mu[1])*v(z, mu[1])*a[1, 1] + u(z, mu[2])*v(z, mu[2])*a[1, 2])*u(z, mu[1])*v(z, mu[1])/2 - (u(z, mu[1])*v(z, mu[1])*a[1, 2] + u(z, mu[2])*v(z, mu[2])*a[2, 2])*u(z, mu[2])*v(z, mu[2])/2 - u(z, mu[1])*v(z, mu[1])*a[1] + u(z, mu[1])*v(z, mu[2]) + u(z, mu[2])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2])*a[2])

In [17]:
Horig = Eq(a[0], solve(a_0_eq.doit(), a[0])[0])

Horig
H_swtch.expand()

Eq(a[0], -u(z, mu[1])**2*v(z, mu[1])**2*a[1, 1]/2 - u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*a[1, 2]/2 - u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*a[2, 1]/2 + u(z, mu[1])*u(z, mu[2]) - u(z, mu[1])*v(z, mu[1])*a[1] - u(z, mu[2])**2*v(z, mu[2])**2*a[2, 2]/2 - u(z, mu[2])*v(z, mu[2])*a[2] + v(z, mu[1])*v(z, mu[2]))

Eq(a[0], -u(z, mu[1])**2*v(z, mu[1])**2*a[1, 1]/2 - u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*a[1, 2] - u(z, mu[1])*v(z, mu[1])*a[1] + u(z, mu[1])*v(z, mu[2]) - u(z, mu[2])**2*v(z, mu[2])**2*a[2, 2]/2 + u(z, mu[2])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2])*a[2])

In [18]:
duv_pow_diff_swtch = Eq(
    Derivative(u(z,mu[1])*v(z, mu[1]) + u(z,mu[2])*v(z, mu[2]), z),
    diff(u(z,mu[1])*v(z, mu[1]) + u(z,mu[2])*v(z, mu[2]), z)
    .subs(du_dv_swtch_subs).simplify()
)

duv_pow_diff_swtch

Eq(Derivative(u(z, mu[1])*v(z, mu[1]) + u(z, mu[2])*v(z, mu[2]), z), 0)

In [19]:
for _ in du_dv_swtch:
    _.subs(_vu2_switch).expand()

Eq(Derivative(u(z, mu[1]), z), -u(z, mu[1])**2*v(z, mu[1])*a[1, 1] - u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*a[1, 2] - u(z, mu[1])*a[1] + v(z, mu[2]))

Eq(Derivative(v(z, mu[2]), z), -u(z, mu[1])*v(z, mu[1])*v(z, mu[2])*a[1, 2] + u(z, mu[1]) - u(z, mu[2])*v(z, mu[2])**2*a[2, 2] - v(z, mu[2])*a[2])

Eq(Derivative(v(z, mu[1]), z), u(z, mu[1])*v(z, mu[1])**2*a[1, 1] + u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*a[1, 2] - u(z, mu[2]) + v(z, mu[1])*a[1])

Eq(Derivative(u(z, mu[2]), z), u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*a[1, 2] + u(z, mu[2])**2*v(z, mu[2])*a[2, 2] + u(z, mu[2])*a[2] - v(z, mu[1]))

In [20]:
_vu2_switchb = [(v(z, mu[2]),x), (u(z, mu[2]), v(z, mu[2])), (x, u(z, mu[2]))]

duj.subs(j,1).doit().expand()
du_dv_swtch[0].subs(_vu2_switchb).expand()

duj.subs(j,2).doit().expand()
du_dv_swtch[3].subs(_vu2_switchb).expand()
# dvj

Eq(Derivative(u(z, mu[1]), z), -u(z, mu[1])**2*v(z, mu[1])*a[1, 1] - u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*a[1, 2] - u(z, mu[1])*a[1] + v(z, mu[2]))

Eq(Derivative(u(z, mu[1]), z), -u(z, mu[1])**2*v(z, mu[1])*a[1, 1] - u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*a[1, 2] - u(z, mu[1])*a[1] + v(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), -u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*a[2, 1] - u(z, mu[2])**2*v(z, mu[2])*a[2, 2] - u(z, mu[2])*a[2] + v(z, mu[1]))

Eq(Derivative(u(z, mu[2]), z), u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*a[1, 2] + u(z, mu[2])**2*v(z, mu[2])*a[2, 2] + u(z, mu[2])*a[2] - v(z, mu[1]))

### Jensen's system

Jensen consider the following model where $\mathrm{Ac}$ denotes $A^*$ the complex conjugate:

In [21]:
jensen = [
    Eq(
        -I*Derivative(A(z, mu[1]), z),
        Omega[1]*A(z, mu[1]) + Omega[2]*A(z, mu[2]) +
        (Omega[3]*A(z, mu[1])*Ac(z, mu[1]) + 2*Omega[4]*A(z, mu[2])*Ac(z, mu[2]))*A(z, mu[1])
    ),
    Eq(
        -I*Derivative(A(z, mu[2]), z),
        Omega[1]*A(z, mu[2]) + Omega[2]*A(z, mu[1]) +
        (Omega[3]*A(z, mu[2])*Ac(z, mu[2]) + 2*Omega[4]*A(z, mu[1])*Ac(z, mu[1]))*A(z, mu[2])
    )
]
jensen[0] = Eq(I*jensen[0].lhs, I*jensen[0].rhs)
jensen[1] = Eq(I*jensen[1].lhs, I*jensen[1].rhs)

_conj = [(A, u), (Ac, A), (u, Ac), (I,-I)]
jensen.append(jensen[0].subs(_conj))
jensen.append(jensen[1].subs(_conj))

jensen_subs = [_.args for _ in jensen]

jens_p0 = Eq(p[0], A(z, mu[1])*Ac(z, mu[1]) + A(z, mu[2])*Ac(z, mu[2]))
jens_p1 = Eq(
    p[1], 
    2*(A(z, mu[1])*Ac(z, mu[2]) + A(z, mu[2])*Ac(z, mu[1])) -
    2*(Omega[3] - 2*Omega[4])/Omega[2]*A(z, mu[1])*Ac(z, mu[1])*A(z, mu[2])*Ac(z, mu[2])
)

assert diff(jens_p0.rhs, z).doit().subs(jensen_subs).simplify() == 0
assert diff(jens_p1.rhs, z).doit().subs(jensen_subs).simplify().collect(Omega[4], factor) == 0

for _ in jensen:
    _
    
jens_p0
jens_p1

Eq(Derivative(A(z, mu[1]), z), I*((A(z, mu[1])*Ac(z, mu[1])*Omega[3] + 2*A(z, mu[2])*Ac(z, mu[2])*Omega[4])*A(z, mu[1]) + A(z, mu[1])*Omega[1] + A(z, mu[2])*Omega[2]))

Eq(Derivative(A(z, mu[2]), z), I*((2*A(z, mu[1])*Ac(z, mu[1])*Omega[4] + A(z, mu[2])*Ac(z, mu[2])*Omega[3])*A(z, mu[2]) + A(z, mu[1])*Omega[2] + A(z, mu[2])*Omega[1]))

Eq(Derivative(Ac(z, mu[1]), z), -I*((A(z, mu[1])*Ac(z, mu[1])*Omega[3] + 2*A(z, mu[2])*Ac(z, mu[2])*Omega[4])*Ac(z, mu[1]) + Ac(z, mu[1])*Omega[1] + Ac(z, mu[2])*Omega[2]))

Eq(Derivative(Ac(z, mu[2]), z), -I*((2*A(z, mu[1])*Ac(z, mu[1])*Omega[4] + A(z, mu[2])*Ac(z, mu[2])*Omega[3])*Ac(z, mu[2]) + Ac(z, mu[1])*Omega[2] + Ac(z, mu[2])*Omega[1]))

Eq(p[0], A(z, mu[1])*Ac(z, mu[1]) + A(z, mu[2])*Ac(z, mu[2]))

Eq(p[1], -(2*Omega[3] - 4*Omega[4])*A(z, mu[1])*A(z, mu[2])*Ac(z, mu[1])*Ac(z, mu[2])/Omega[2] + 2*A(z, mu[1])*Ac(z, mu[2]) + 2*A(z, mu[2])*Ac(z, mu[1]))

If we normalise the length scale so the wave mixing term is equal to 1 then we can obtain Jensen's system as folows:

In [22]:
my_subs = [
    (A(z, mu[1]), u(z, mu[1])*exp(I*pi/4)), 
    (A(z, mu[2]), v(z, mu[2])*exp(-I*pi/4)),
    (Ac(z, mu[1]), -v(z, mu[1])*exp(-I*pi/4)),
    (Ac(z, mu[2]), u(z, mu[2])*exp(I*pi/4)),
    (Omega[2], 1)
]

_eq_ = jensen[2].subs(my_subs).expand().doit()
_eq_ = Eq(-_eq_.lhs, -_eq_.rhs)


jensen_to_uv = [
    Eq(
        diff(u(z, mu[1]),z), 
        solve(jensen[0].subs(my_subs).expand().doit(), diff(u(z, mu[1]),z))[0]
        .expand()
    ),
    Eq(
        diff(u(z, mu[2]),z), 
        solve(jensen[3].subs(my_subs).expand().doit(), diff(u(z, mu[2]),z))[0]
        .expand()
    ),
    Eq(
        diff(v(z, mu[1]),z), 
        solve(jensen[2].subs(my_subs).expand().doit(), diff(v(z, mu[1]),z))[0]
        .expand()
    ),
    Eq(
        diff(v(z, mu[2]),z), 
        solve(jensen[1].subs(my_subs).expand().doit(), diff(v(z, mu[2]),z))[0]
        .expand()
    )
]

for _ in my_subs:
    Eq(*_)
print()
for _ in jensen_to_uv:
    _

Eq(A(z, mu[1]), u(z, mu[1])*exp(I*pi/4))

Eq(A(z, mu[2]), v(z, mu[2])*exp(-I*pi/4))

Eq(Ac(z, mu[1]), -v(z, mu[1])*exp(-I*pi/4))

Eq(Ac(z, mu[2]), u(z, mu[2])*exp(I*pi/4))

Eq(Omega[2], 1)

Eq(Derivative(u(z, mu[1]), z), -I*u(z, mu[1])**2*v(z, mu[1])*Omega[3] + 2*I*u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*Omega[4] + I*u(z, mu[1])*Omega[1] + v(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), 2*I*u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*Omega[4] - I*u(z, mu[2])**2*v(z, mu[2])*Omega[3] - I*u(z, mu[2])*Omega[1] + v(z, mu[1]))

Eq(Derivative(v(z, mu[1]), z), I*u(z, mu[1])*v(z, mu[1])**2*Omega[3] - 2*I*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*Omega[4] - u(z, mu[2]) - I*v(z, mu[1])*Omega[1])

Eq(Derivative(v(z, mu[2]), z), -2*I*u(z, mu[1])*v(z, mu[1])*v(z, mu[2])*Omega[4] - u(z, mu[1]) + I*u(z, mu[2])*v(z, mu[2])**2*Omega[3] + I*v(z, mu[2])*Omega[1])

In [23]:
my_subs_b = [
    (a[1], -I*Omega[1]),
    (a[2], I*Omega[1]),
    (a[1,1], I*Omega[3]),
    (a[2,2], I*Omega[3]),
    (a[1,2], -2*I*Omega[4]),
    (a[2,1], -2*I*Omega[4])
]

uv_to_jensen = [
    duj.subs(j,1).doit().expand().subs(my_subs_b),
    duj.subs(j,2).doit().expand().subs(my_subs_b),
    dvj.subs(j,1).doit().expand().subs(my_subs_b),
    dvj.subs(j,2).doit().expand().subs(my_subs_b)
]

for _j in range(4):
    assert (jensen_to_uv[_j].rhs - uv_to_jensen[_j].rhs).factor().simplify() == 0, 'not equivalent to jensen'

for _ in my_subs_b:
    Eq(*_)
print()
for _ in uv_to_jensen:
    _

Eq(a[1], -I*Omega[1])

Eq(a[2], I*Omega[1])

Eq(a[1, 1], I*Omega[3])

Eq(a[2, 2], I*Omega[3])

Eq(a[1, 2], -2*I*Omega[4])

Eq(a[2, 1], -2*I*Omega[4])

Eq(Derivative(u(z, mu[1]), z), -I*u(z, mu[1])**2*v(z, mu[1])*Omega[3] + 2*I*u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*Omega[4] + I*u(z, mu[1])*Omega[1] + v(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), 2*I*u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*Omega[4] - I*u(z, mu[2])**2*v(z, mu[2])*Omega[3] - I*u(z, mu[2])*Omega[1] + v(z, mu[1]))

Eq(Derivative(v(z, mu[1]), z), I*u(z, mu[1])*v(z, mu[1])**2*Omega[3] - 2*I*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*Omega[4] - u(z, mu[2]) - I*v(z, mu[1])*Omega[1])

Eq(Derivative(v(z, mu[2]), z), -2*I*u(z, mu[1])*v(z, mu[1])*v(z, mu[2])*Omega[4] - u(z, mu[1]) + I*u(z, mu[2])*v(z, mu[2])**2*Omega[3] + I*v(z, mu[2])*Omega[1])

More generally, without such a length normalisation, we would do:

In [24]:
my_subs_c = [
    (a[1], -I*Omega[1]/Omega[2]),
    (a[2], I*Omega[1]/Omega[2]),
    (a[1,1], I*Omega[3]/Omega[2]),
    (a[2,2], I*Omega[3]/Omega[2]),
    (a[1,2], -I*2*Omega[4]/Omega[2]),
    (a[2,1], -I*2*Omega[4]/Omega[2]),
    (u(z, mu[1]), A(z/Omega[2], mu[1])*exp(-I*pi/4)), 
    (v(z, mu[2]), A(z/Omega[2], mu[2])*exp(I*pi/4)),
    (v(z, mu[1]), -Ac(z/Omega[2], mu[1])*exp(I*pi/4)),
    (u(z, mu[2]), Ac(z/Omega[2], mu[2])*exp(-I*pi/4))
]

uv_to_jensen_A = [
    duj.subs(j,1).doit().expand().subs(my_subs_c).subs(z, x*Omega[2]).doit(),
    duj.subs(j,2).doit().expand().subs(my_subs_c).subs(z, x*Omega[2]).doit(),
    dvj.subs(j,1).doit().expand().subs(my_subs_c).subs(z, x*Omega[2]).doit(),
    dvj.subs(j,2).doit().expand().subs(my_subs_c).subs(z, x*Omega[2]).doit()
]
uv_to_jensen_A = [
    Eq(diff(A(x, mu[1]),x), solve(uv_to_jensen_A[0], diff(A(x, mu[1]),x))[0].expand()),
    Eq(diff(A(x, mu[2]),x), solve(uv_to_jensen_A[3], diff(A(x, mu[2]),x))[0].expand()),
    Eq(diff(Ac(x, mu[1]),x), solve(uv_to_jensen_A[2], diff(Ac(x, mu[1]),x))[0].expand()),
    Eq(diff(Ac(x, mu[2]),x), solve(uv_to_jensen_A[1], diff(Ac(x, mu[2]),x))[0].expand())
]

duj
dvj
print()

for _j in range(4):
    assert (jensen[_j].rhs - uv_to_jensen_A[_j].rhs.subs(x,z)).factor().simplify() == 0, 'not equivalent to Jensen'

for _ in my_subs_c:
    Eq(*_)
print()
for _ in uv_to_jensen_A:
    _

Eq(Derivative(u(z, mu[j]), z), -(a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*u(z, mu[j]) + Product(v(z, mu[k]), (k, 1, 2))/v(z, mu[j]))

Eq(Derivative(v(z, mu[j]), z), (a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*v(z, mu[j]) - Product(u(z, mu[k]), (k, 1, 2))/u(z, mu[j]))

Eq(a[1], -I*Omega[1]/Omega[2])

Eq(a[2], I*Omega[1]/Omega[2])

Eq(a[1, 1], I*Omega[3]/Omega[2])

...

Eq(u(z, mu[2]), Ac(z/Omega[2], mu[2])*exp(-I*pi/4))

Eq(Derivative(A(x, mu[1]), x), I*A(x, mu[1])**2*Ac(x, mu[1])*Omega[3] + 2*I*A(x, mu[1])*A(x, mu[2])*Ac(x, mu[2])*Omega[4] + I*A(x, mu[1])*Omega[1] + I*A(x, mu[2])*Omega[2])

Eq(Derivative(A(x, mu[2]), x), 2*I*A(x, mu[1])*A(x, mu[2])*Ac(x, mu[1])*Omega[4] + I*A(x, mu[1])*Omega[2] + I*A(x, mu[2])**2*Ac(x, mu[2])*Omega[3] + I*A(x, mu[2])*Omega[1])

Eq(Derivative(Ac(x, mu[1]), x), -I*A(x, mu[1])*Ac(x, mu[1])**2*Omega[3] - 2*I*A(x, mu[2])*Ac(x, mu[1])*Ac(x, mu[2])*Omega[4] - I*Ac(x, mu[1])*Omega[1] - I*Ac(x, mu[2])*Omega[2])

Eq(Derivative(Ac(x, mu[2]), x), -2*I*A(x, mu[1])*Ac(x, mu[1])*Ac(x, mu[2])*Omega[4] - I*A(x, mu[2])*Ac(x, mu[2])**2*Omega[3] - I*Ac(x, mu[1])*Omega[2] - I*Ac(x, mu[2])*Omega[1])

In [94]:
b_j_coeffs_subs = [_.args for _ in b_j_coeffs]
c_j_coeffs_subs = [_.args for _ in c_j_coeffs]
d_j_coeffs_subs = [_.args for _ in c_j_coeffs]
d_j_coeffs_jensen = [
    _.subs(c_j_coeffs_subs).subs(b_j_coeffs_subs).doit().subs(gamma[2], -gamma[1]).subs(my_subs_c).simplify()
    for _ in d_j_coeffs
]
d_j_coeffs_jensen_subs = [_.args for _ in d_j_coeffs_jensen]
sqrt_d4_b2 = Eq(sqrt(d[4]), b[2])
Lam1_h = Lams[1].doit().subs(m,1).subs(my_subs_c)
Lam2_h = Lams[1].doit().subs(m,2).subs(my_subs_c)


for _ in d_j_coeffs_jensen:
    _
g2_dj.subs([(d[1],0),(d[3],0)])
g3_dj.subs([(d[1],0),(d[3],0)])

for _ in b_j_coeffs:
    _.doit().subs(gamma[2], - gamma[1]).subs(my_subs_c).simplify()
    

_r_j_h_subs = [
    sqrt_d4_b2.args, 
    Lam1_h.args,
    Lam2_h.args, 
    b_j_coeffs[2].doit().subs(gamma[2], - gamma[1]).subs(my_subs_c).args
]

_r2_subs_Jensen = [
    r2j_log_part.subs(j,_j+1).subs(_r_j_h_subs).simplify().args for _j in range(2)
]
jensen_r2j = Eq(r[2,j], 0)

Lam1_h
Lam2_h
assert (Lam1_h.rhs*2 - b_j_coeffs[2].doit().subs(gamma[2], - gamma[1]).subs(my_subs_c).rhs).simplify() == 0
assert (Lam2_h.rhs*2 - b_j_coeffs[2].doit().subs(gamma[2], - gamma[1]).subs(my_subs_c).rhs).simplify() == 0
Eq(Lamda[1,j]*2, b[2])

for _j, _ in enumerate(_r2_subs_Jensen):
    assert jensen_r2j.subs(j,_j+1).subs(*_)

r1j_log_part.subs(j,1).subs(_r_j_h_subs).simplify()
r1j_log_part.subs(j,2).subs(_r_j_h_subs).simplify()
r2j_log_part.subs(j,1).subs(_r_j_h_subs).simplify()
r2j_log_part.subs(j,2).subs(_r_j_h_subs).simplify()

Eq(d[0], ((I*(Omega[3] + 2*Omega[4])*gamma[1]**2 - 2*I*Omega[1]*gamma[1] + Omega[2]*a[0])**2 + 4*Omega[2]**2*gamma[1]**2)/Omega[2]**2)

Eq(d[1], 0)

Eq(d[2], 2*(I*(Omega[3] - 2*Omega[4])*(I*(Omega[3] + 2*Omega[4])*gamma[1]**2 - 2*I*Omega[1]*gamma[1] + Omega[2]*a[0]) - 2*Omega[2]**2)/Omega[2]**2)

Eq(d[3], 0)

Eq(d[4], -(Omega[3] - 2*Omega[4])**2/Omega[2]**2)

...

Eq(2*Lamda[1, j], b[2])

Eq(r[1, 1], 1/2)

Eq(r[1, 2], 1/2)

Eq(r[2, 1], 0)

Eq(r[2, 2], 0)

In [106]:
Lam01_jensen = Lams[0].doit().subs(m,1).subs(gamma[2], - gamma[1]).subs(my_subs_c).simplify()
Lam01_jensen = Eq(Lam01_jensen.lhs, Lam01_jensen.rhs.expand().collect(gamma[1], factor))

Lam02_jensen = Lams[0].doit().subs(m,2).subs(gamma[1], - gamma[2]).subs(my_subs_c).simplify()
Lam02_jensen = Eq(Lam02_jensen.lhs, Lam02_jensen.rhs.expand().collect(gamma[2], factor))

Lam01_jensen
Lam02_jensen

Eq(Lamda[0, 1], -I*(3*Omega[3] + 2*Omega[4])*gamma[1]/(2*Omega[2]) + I*Omega[1]/Omega[2])

Eq(Lamda[0, 2], -I*(3*Omega[3] + 2*Omega[4])*gamma[2]/(2*Omega[2]) - I*Omega[1]/Omega[2])

In [57]:
uz_sol_no_sqrt_Jensen = uz_sol_no_sqrt.subs([jensen_r2j.args])
vz_sol_no_sqrt_Jensen = vz_sol_no_sqrt.subs([jensen_r2j.args])

uz_sol_no_sqrt_Jensen
vz_sol_no_sqrt_Jensen

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 - z1, g2, g3)*sqrt(b[2])))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 + z1, g2, g3)*sqrt(b[2])))

In [81]:
_eps_to_kap = [
    (exp(epsilon[1]), kappa[1]/exp(I*pi/4)),
    (exp(epsilon[2]), kappa[2]/exp(I*pi/4))
]

A_Jensen_sigs = [
    Eq(
        my_subs_c[6][1].subs(z,z*Omega[2])*exp(I*pi/4), 
        (my_subs_c[6][0].subs([uz_sol_no_sqrt_Jensen.subs(j,1).args]).subs(z,z*Omega[2])*exp(I*pi/4))
        .expand().subs(_eps_to_kap)
    ),
    Eq(
        my_subs_c[7][1].subs(z,z*Omega[2])*exp(-I*pi/4), 
        (my_subs_c[7][0].subs([vz_sol_no_sqrt_Jensen.subs(j,2).args]).subs(z,z*Omega[2])*exp(-I*pi/4))
        .expand().subs(_eps_to_kap)
    ),
    Eq(
        -my_subs_c[8][1].subs(z,z*Omega[2])*exp(-I*pi/4), 
        (-my_subs_c[8][0].subs([vz_sol_no_sqrt_Jensen.subs(j,1).args]).subs(z,z*Omega[2])*exp(-I*pi/4))
        .expand().subs(_eps_to_kap)
    ),
    Eq(
        my_subs_c[9][1].subs(z,z*Omega[2])*exp(I*pi/4), 
        (my_subs_c[9][0].subs([uz_sol_no_sqrt_Jensen.subs(j,2).args]).subs(z,z*Omega[2])*exp(I*pi/4))
        .expand().subs(_eps_to_kap)
    )
]

for _ in A_Jensen_sigs:
    _

Eq(A(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z*Omega[2] - 2*z0 + mu[1], g2, g3)*exp(z*Omega[2]*r[0, 1])*kappa[1]/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z*Omega[2] - z0 - z1, g2, g3)*sqrt(b[2])))

Eq(A(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z*Omega[2] - mu[2], g2, g3)*exp(-z*Omega[2]*r[0, 2])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z*Omega[2] - z0 + z1, g2, g3)*sqrt(b[2])*kappa[2]))

Eq(Ac(z, mu[1]), -sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z*Omega[2] - mu[1], g2, g3)*exp(-z*Omega[2]*r[0, 1])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z*Omega[2] - z0 + z1, g2, g3)*sqrt(b[2])*kappa[1]))

Eq(Ac(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z*Omega[2] - 2*z0 + mu[2], g2, g3)*exp(z*Omega[2]*r[0, 2])*kappa[2]/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z*Omega[2] - z0 - z1, g2, g3)*sqrt(b[2])))

### Jensen's system with linear diagonalisation of wave mixing

In this section we show that the Jensen coherent coupler can be transformed into a degeneration of a FWM system, namely that of nonlinear polarisation in which the four modes form 2 pairs. For Jensen's parameters this kills the quartic term hence Jensen's is a special case.

In [17]:
MM = Matrix([[Omega[1], Omega[2]], [Omega[2], Omega[1]]])
PP, DD = MM.diagonalize()
PPi = PP**(-1)
PP = 1/sqrt(2)*PP
PPi = sqrt(2)*PPi

AA = Matrix([A(z, mu[1]), A(z, mu[2])])
AAc = Matrix([Ac(z, mu[1]), Ac(z, mu[2])])
BB = Matrix([B(z, mu[1]), B(z, mu[2])])
BBc = Matrix([Bc(z, mu[1]), Bc(z, mu[2])])

B_as_A = Eq(BB, PPi*AA)
Bc_as_Ac = Eq(BBc, PPi*AAc)
A_as_B = Eq(AA, PP*BB)
Ac_as_Bc = Eq(AAc, PP*BBc)

A_as_B_subs = [
    (A_as_B.lhs[0], A_as_B.rhs[0]), 
    (A_as_B.lhs[1], A_as_B.rhs[1])
]
Ac_as_Bc_subs = [
    (Ac_as_Bc.lhs[0], Ac_as_Bc.rhs[0]), 
    (Ac_as_Bc.lhs[1], Ac_as_Bc.rhs[1])
]
all_A_as_B_subs = A_as_B_subs + Ac_as_Bc_subs
B_as_A_subs = [
    (B_as_A.lhs[0], B_as_A.rhs[0]), 
    (B_as_A.lhs[1], B_as_A.rhs[1])
]
Bc_as_Ac_subs = [
    (Bc_as_Ac.lhs[0], Bc_as_Ac.rhs[0]), 
    (Bc_as_Ac.lhs[1], Bc_as_Ac.rhs[1])
]

assert B_as_A.subs(A_as_B_subs).simplify()
assert Bc_as_Ac.subs(Ac_as_Bc_subs).simplify()
assert A_as_B.subs(B_as_A_subs).simplify()
assert Ac_as_Bc.subs(Bc_as_Ac_subs).simplify()

MM
PP
PPi
DD
B_as_A
Bc_as_Ac

Matrix([
[Omega[1], Omega[2]],
[Omega[2], Omega[1]]])

Matrix([
[-sqrt(2)/2, sqrt(2)/2],
[ sqrt(2)/2, sqrt(2)/2]])

Matrix([
[-sqrt(2)/2, sqrt(2)/2],
[ sqrt(2)/2, sqrt(2)/2]])

Matrix([
[Omega[1] - Omega[2],                   0],
[                  0, Omega[1] + Omega[2]]])

Eq(Matrix([
[B(z, mu[1])],
[B(z, mu[2])]]), Matrix([
[-sqrt(2)*A(z, mu[1])/2 + sqrt(2)*A(z, mu[2])/2],
[ sqrt(2)*A(z, mu[1])/2 + sqrt(2)*A(z, mu[2])/2]]))

Eq(Matrix([
[Bc(z, mu[1])],
[Bc(z, mu[2])]]), Matrix([
[-sqrt(2)*Ac(z, mu[1])/2 + sqrt(2)*Ac(z, mu[2])/2],
[ sqrt(2)*Ac(z, mu[1])/2 + sqrt(2)*Ac(z, mu[2])/2]]))

In [18]:
B_jensen = [_.subs(x,z).subs(all_A_as_B_subs).doit() for _ in uv_to_jensen_A]
B_diffs = [diff(B(z, mu[1]),z), diff(B(z, mu[2]),z), diff(Bc(z, mu[1]),z), diff(Bc(z, mu[2]),z)]
B_jensen_sols = solve(B_jensen, B_diffs)
dB_jensen = [Eq(_, B_jensen_sols[_]) for _ in B_diffs]

_collects =[
    [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2]), B(z, mu[1])],
    [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2]), B(z, mu[2])],
    [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2]), Bc(z, mu[1])],
    [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2]), Bc(z, mu[2])]
]

Omega_beta_subs = [
    (Omega[1] - Omega[2], psi[0]),
    (Omega[1] + Omega[2], psi[1]),
    (Omega[3] - 2*Omega[4], 2*psi[2]),
    (Omega[3] + 2*Omega[4], 2*psi[3]),
    (Omega[3], psi[4])
]

dB_jensen = [
    Eq(
        _eq.lhs, 
        _eq.rhs
        .expand().collect(_c, factor)
        .subs(Omega_beta_subs)
        .expand().collect([psi[0], psi[1], psi[2]], factor)
    ) 
    for _c, _eq in zip(_collects, dB_jensen)
]

for _ in Omega_beta_subs:
    Eq(*_)
print()
for _ in dB_jensen:
    _

Eq(Omega[1] - Omega[2], psi[0])

Eq(Omega[1] + Omega[2], psi[1])

Eq(Omega[3] - 2*Omega[4], 2*psi[2])

Eq(Omega[3] + 2*Omega[4], 2*psi[3])

Eq(Omega[3], psi[4])

Eq(Derivative(B(z, mu[1]), z), I*(B(z, mu[1])*Bc(z, mu[1])*psi[3] + B(z, mu[2])*Bc(z, mu[2])*psi[4])*B(z, mu[1]) + I*B(z, mu[1])*psi[0] + I*B(z, mu[2])**2*Bc(z, mu[1])*psi[2])

Eq(Derivative(B(z, mu[2]), z), I*(B(z, mu[1])*Bc(z, mu[1])*psi[4] + B(z, mu[2])*Bc(z, mu[2])*psi[3])*B(z, mu[2]) + I*B(z, mu[1])**2*Bc(z, mu[2])*psi[2] + I*B(z, mu[2])*psi[1])

Eq(Derivative(Bc(z, mu[1]), z), -I*(B(z, mu[1])*Bc(z, mu[1])*psi[3] + B(z, mu[2])*Bc(z, mu[2])*psi[4])*Bc(z, mu[1]) - I*B(z, mu[1])*Bc(z, mu[2])**2*psi[2] - I*Bc(z, mu[1])*psi[0])

Eq(Derivative(Bc(z, mu[2]), z), -I*(B(z, mu[1])*Bc(z, mu[1])*psi[4] + B(z, mu[2])*Bc(z, mu[2])*psi[3])*Bc(z, mu[2]) - I*B(z, mu[2])*Bc(z, mu[1])**2*psi[2] - I*Bc(z, mu[2])*psi[1])

In [19]:
jens_H_B = Eq(
    p[2], 
    I*psi[3]/2*Sum((B(z, mu[j])*Bc(z, mu[j]))**2, (j, 1, 2)) +
    I*psi[4]*(B(z, mu[1])*Bc(z, mu[1])*B(z, mu[2])*Bc(z, mu[2])) +
    I*psi[0]*B(z, mu[1])*Bc(z, mu[1]) + I*psi[1]*B(z, mu[2])*Bc(z, mu[2]) +
    I*psi[2]/2*(B(z, mu[1])**2*Bc(z, mu[2])**2 + Bc(z, mu[1])**2*B(z, mu[2])**2)
)

jens_H_B_wm = Eq(
    -2*(jens_H_B.lhs - jens_H_B.lhs - jens_H_B.rhs.coeff(psi[2],1)*psi[2]),
    -2*(jens_H_B.rhs - jens_H_B.lhs - jens_H_B.rhs.coeff(psi[2],1)*psi[2])
)

dB_jensen_from_H = [
    Eq(diff(B(z, mu[1]),z), diff(jens_H_B.rhs.doit(), Bc(z, mu[1])).collect(_collects[0], factor)),
    Eq(diff(B(z, mu[2]),z), diff(jens_H_B.rhs.doit(), Bc(z, mu[2])).collect(_collects[1], factor)),
    Eq(diff(Bc(z, mu[1]),z), -diff(jens_H_B.rhs.doit(), B(z, mu[1])).collect(_collects[2], factor)),
    Eq(diff(Bc(z, mu[2]),z), -diff(jens_H_B.rhs.doit(), B(z, mu[2])).collect(_collects[3], factor))
]
dB_jensen_from_H_subs = [_.args for _ in dB_jensen_from_H]

for _j in range(4):
    assert (dB_jensen_from_H[_j].rhs - dB_jensen[_j].rhs).simplify() == 0, 'H not yielding right diff eqns'
    
assert diff(jens_H_B.rhs,z).doit().subs(dB_jensen_from_H_subs).simplify() ==0, 'H not conserved'

jens_p0_B = jens_p0.subs(all_A_as_B_subs).simplify()

jens_H_B
jens_p0_B

Eq(p[2], I*(B(z, mu[1])**2*Bc(z, mu[2])**2 + B(z, mu[2])**2*Bc(z, mu[1])**2)*psi[2]/2 + I*B(z, mu[1])*B(z, mu[2])*Bc(z, mu[1])*Bc(z, mu[2])*psi[4] + I*B(z, mu[1])*Bc(z, mu[1])*psi[0] + I*B(z, mu[2])*Bc(z, mu[2])*psi[1] + I*psi[3]*Sum(B(z, mu[j])**2*Bc(z, mu[j])**2, (j, 1, 2))/2)

Eq(p[0], B(z, mu[1])*Bc(z, mu[1]) + B(z, mu[2])*Bc(z, mu[2]))

In [20]:
rhoB12 = Eq(rho(z), B(z, mu[1])*Bc(z, mu[1]) - B(z, mu[2])*Bc(z, mu[2]))
B_sols_rho_p0 = solve([rhoB12, jens_p0_B], [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2])])
B_sols_rho_p0_eqs = [Eq(_k, B_sols_rho_p0[_k]) for _k in B_sols_rho_p0]

dB1 = Eq(
    Derivative(B(z, mu[1])*Bc(z, mu[1]),z),
    Derivative(B(z, mu[1])*Bc(z, mu[1]),z).doit().subs(dB_jensen_from_H_subs).simplify()
)
dB2 = Eq(
    Derivative(B(z, mu[2])*Bc(z, mu[2]),z),
    Derivative(B(z, mu[2])*Bc(z, mu[2]),z).doit().subs(dB_jensen_from_H_subs).simplify()
)

rhoB12
for _ in B_sols_rho_p0_eqs:
    _
print()
dB1
dB2

Eq(rho(z), B(z, mu[1])*Bc(z, mu[1]) - B(z, mu[2])*Bc(z, mu[2]))

Eq(B(z, mu[1])*Bc(z, mu[1]), rho(z)/2 + p[0]/2)

Eq(B(z, mu[2])*Bc(z, mu[2]), -rho(z)/2 + p[0]/2)

Eq(Derivative(B(z, mu[1])*Bc(z, mu[1]), z), I*(-B(z, mu[1])**2*Bc(z, mu[2])**2 + B(z, mu[2])**2*Bc(z, mu[1])**2)*psi[2])

Eq(Derivative(B(z, mu[2])*Bc(z, mu[2]), z), I*(B(z, mu[1])**2*Bc(z, mu[2])**2 - B(z, mu[2])**2*Bc(z, mu[1])**2)*psi[2])

In [21]:
drho_B_sqrd = Eq(
    4*dB1.lhs.subs(B_sols_rho_p0).doit()**2,
    4*(dB1.rhs**2 - jens_H_B_wm.lhs**2 + jens_H_B_wm.rhs**2)
    .doit().expand()
    .subs(B_sols_rho_p0)
    .expand().collect(rho(z), factor)
)

d_B_coeff_subs = [(drho_B_sqrd.rhs.coeff(rho(z), _j).factor(), d[_j]) for _j in range(5)]
d_B_coeffs = [Eq(_[1], _[0]) for _ in d_B_coeff_subs]
drho_B_sqrd = drho_B_sqrd.subs(d_B_coeff_subs)

drho_B_sqrd

for _ in d_B_coeffs:
    _

Eq(Derivative(rho(z), z)**2, rho(z)**4*d[4] + rho(z)**3*d[3] + rho(z)**2*d[2] + rho(z)*d[1] + d[0])

Eq(d[0], (p[0]**2*psi[2] - p[0]**2*psi[3] - p[0]**2*psi[4] - 2*p[0]*psi[0] - 2*p[0]*psi[1] - 4*I*p[2])*(p[0]**2*psi[2] + p[0]**2*psi[3] + p[0]**2*psi[4] + 2*p[0]*psi[0] + 2*p[0]*psi[1] + 4*I*p[2]))

Eq(d[1], -4*(psi[0] - psi[1])*(p[0]**2*psi[3] + p[0]**2*psi[4] + 2*p[0]*psi[0] + 2*p[0]*psi[1] + 4*I*p[2]))

Eq(d[2], -2*(p[0]**2*psi[2]**2 + p[0]**2*psi[3]**2 - p[0]**2*psi[4]**2 + 2*p[0]*psi[0]*psi[3] - 2*p[0]*psi[0]*psi[4] + 2*p[0]*psi[1]*psi[3] - 2*p[0]*psi[1]*psi[4] + 4*I*p[2]*psi[3] - 4*I*p[2]*psi[4] + 2*psi[0]**2 - 4*psi[0]*psi[1] + 2*psi[1]**2))

Eq(d[3], -4*(psi[0] - psi[1])*(psi[3] - psi[4]))

Eq(d[4], (psi[2] - psi[3] + psi[4])*(psi[2] + psi[3] - psi[4]))

In [22]:
psi_Omega_subs = [(_[1], _[0]) for _ in Omega_beta_subs]
psi_Omega_subs[2] = (psi_Omega_subs[2][0]/2, psi_Omega_subs[2][1]/2)
psi_Omega_subs[3] = (psi_Omega_subs[3][0]/2, psi_Omega_subs[3][1]/2)

d_B_coeffs_Omega = [Eq(_.lhs, _.rhs.subs(psi_Omega_subs).factor()) for _ in d_B_coeffs]

for _ in d_B_coeffs_Omega:
    _

Eq(d[0], -2*(2*Omega[1]*p[0] + Omega[3]*p[0]**2 + 2*I*p[2])*(4*Omega[1]*p[0] + Omega[3]*p[0]**2 + 2*Omega[4]*p[0]**2 + 4*I*p[2]))

Eq(d[1], 4*(8*Omega[1]*p[0] + 3*Omega[3]*p[0]**2 + 2*Omega[4]*p[0]**2 + 8*I*p[2])*Omega[2])

Eq(d[2], 4*Omega[1]*Omega[3]*p[0] - 8*Omega[1]*Omega[4]*p[0] - 16*Omega[2]**2 + Omega[3]**2*p[0]**2 + 4*I*Omega[3]*p[2] - 4*Omega[4]**2*p[0]**2 - 8*I*Omega[4]*p[2])

Eq(d[3], -4*(Omega[3] - 2*Omega[4])*Omega[2])

Eq(d[4], 0)

In [23]:
rho_w_B = Eq(rho(z), c[1]*w(z) + c[0])

drho_B_sqrd_w = Eq(
    drho_B_sqrd.lhs
    .subs([rho_w_B.args]).doit()/c[1]**2,
    (
        drho_B_sqrd.rhs
        .subs([d_B_coeffs_Omega[4].args])
        .subs(rho(z), c[1]*w(z) + c[0]).doit()/c[1]**2
    ).expand().collect(w(z), factor)
)
c_subs_w_B = [
    Eq(c[0], solve(Eq(drho_B_sqrd_w.rhs.coeff(w(z), 2), 0),c[0])[0]).args, 
    Eq(c[1], solve(Eq(drho_B_sqrd_w.rhs.coeff(w(z), 3), 4), c[1])[0]).args
]

drho_B_sqrd_w = Eq(
    drho_B_sqrd_w.lhs, 
    drho_B_sqrd_w.rhs.subs(c_subs_w_B).expand().collect(w(z), factor)
)
rho_w_B = rho_w_B.subs(c_subs_w_B)

g2_B = Eq(g2, -drho_B_sqrd_w.rhs.coeff(w(z), 1).factor())
g3_B = Eq(g3, -drho_B_sqrd_w.rhs.coeff(w(z), 0).expand())
drho_B_sqrd_w = drho_B_sqrd_w.subs([(-g2_B.rhs, -g2_B.lhs), (-g3_B.rhs, -g3_B.lhs)])


rho_w_B
drho_B_sqrd_w
g2_B
g3_B

Eq(rho(z), 4*w(z)/d[3] - d[2]/(3*d[3]))

Eq(Derivative(w(z), z)**2, -g2*w(z) - g3 + 4*w(z)**3)

Eq(g2, -(3*d[1]*d[3] - d[2]**2)/12)

Eq(g3, -d[0]*d[3]**2/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

### The linear diagonalisation of wave mixing in the general coupler system

In this section we show that trying a linear diagonlisation of the general case for which $b_1\ne0$ does not yield a simple case with no quartic but rather ir creates many new wave mixing terms that are difficult to handle. This motivates the utility of our nonlinear transformations as the appropriate path to explore relations among systems in the general case.

We start from the following system:

In [24]:
du_dv_hat_eqs = [
    Eq(
        Derivative(uhat(z, mu[1]), z),
        -uhat(z, mu[1])**2*vhat(z, mu[1])*b[2] + uhat(z, mu[1])*b[1]/2 + vhat(z, mu[2])
    ),
    Eq(
        Derivative(uhat(z, mu[2]), z),
        -uhat(z, mu[2])**2*vhat(z, mu[2])*b[2] + uhat(z, mu[2])*b[1]/2 + vhat(z, mu[1])
    ),
    Eq(
        Derivative(vhat(z, mu[1]), z),
        uhat(z, mu[1])*vhat(z, mu[1])**2*b[2] - uhat(z, mu[2]) - vhat(z, mu[1])*b[1]/2
    ),
    Eq(
        Derivative(vhat(z, mu[2]), z),
        -uhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2])**2*b[2] - vhat(z, mu[2])*b[1]/2
    )
]

for _ in du_dv_hat_eqs:
    _

Eq(Derivative(uhat(z, mu[1]), z), -uhat(z, mu[1])**2*vhat(z, mu[1])*b[2] + uhat(z, mu[1])*b[1]/2 + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), -uhat(z, mu[2])**2*vhat(z, mu[2])*b[2] + uhat(z, mu[2])*b[1]/2 + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), uhat(z, mu[1])*vhat(z, mu[1])**2*b[2] - uhat(z, mu[2]) - vhat(z, mu[1])*b[1]/2)

Eq(Derivative(vhat(z, mu[2]), z), -uhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2])**2*b[2] - vhat(z, mu[2])*b[1]/2)

The following linear transformation removes the linear wave mixing coupling but at the expense of generating many new FWM terms:

In [25]:
MMh = Matrix([[b[1]/2, 1], [-1, -b[1]/2]])
PPh, DDh = MMh.diagonalize()
PPhi = PPh**(-1)
_det_const_h = sqrt(det(PPh))
PPh = 1/_det_const_h*PPh
PPhi = _det_const_h*PPhi

Uhat_vec1 = Matrix([uhat(z, mu[1]), vhat(z, mu[2])])
Uhat_vec2 = Matrix([uhat(z, mu[2]), vhat(z, mu[1])])
Utilde_vec1 = Matrix([utilde(z, mu[1]), utilde(z, mu[2])])
Utilde_vec2 = Matrix([vtilde(z, mu[1]), vtilde(z, mu[2])])
Utilde1_as_Uhat1 = Eq(Utilde_vec1, PPhi*Uhat_vec1)
Utilde2_as_Uhat2 = Eq(Utilde_vec2, PPhi*Uhat_vec2)
Uhat1_as_Utilde1 = Eq(Uhat_vec1, PPh*Utilde_vec1)
Uhat2_as_Utilde2 = Eq(Uhat_vec2, PPh*Utilde_vec2)

hat_as_tilde_1_subs = [
    (Uhat1_as_Utilde1.lhs[0], Uhat1_as_Utilde1.rhs[0]), 
    (Uhat1_as_Utilde1.lhs[1], Uhat1_as_Utilde1.rhs[1])
]
hat_as_tilde_2_subs = [
    (Uhat2_as_Utilde2.lhs[0], Uhat2_as_Utilde2.rhs[0]), 
    (Uhat2_as_Utilde2.lhs[1], Uhat2_as_Utilde2.rhs[1])
]
all_hat_as_tilde_subs = hat_as_tilde_1_subs + hat_as_tilde_2_subs

tilde_as_hat_1_subs = [
    (Utilde1_as_Uhat1.lhs[0], Utilde1_as_Uhat1.rhs[0]), 
    (Utilde1_as_Uhat1.lhs[1], Utilde1_as_Uhat1.rhs[1])
]
tilde_as_hat_2_subs = [
    (Utilde2_as_Uhat2.lhs[0], Utilde2_as_Uhat2.rhs[0]), 
    (Utilde2_as_Uhat2.lhs[1], Utilde2_as_Uhat2.rhs[1])
]


assert Utilde1_as_Uhat1.subs(hat_as_tilde_1_subs).simplify()
assert Utilde2_as_Uhat2.subs(hat_as_tilde_2_subs).simplify()
assert Uhat1_as_Utilde1.subs(tilde_as_hat_1_subs).simplify()
assert Uhat2_as_Utilde2.subs(tilde_as_hat_2_subs).simplify()

MMh
PPh
PPhi
DDh
Utilde1_as_Uhat1
Utilde2_as_Uhat2
Uhat1_as_Utilde1
Uhat2_as_Utilde2

Matrix([
[b[1]/2,       1],
[    -1, -b[1]/2]])

Matrix([
[(sqrt(b[1]**2 - 4)/2 - b[1]/2)/(b[1]**2 - 4)**(1/4), (-sqrt(b[1]**2 - 4)/2 - b[1]/2)/(b[1]**2 - 4)**(1/4)],
[                              (b[1]**2 - 4)**(-1/4),                                (b[1]**2 - 4)**(-1/4)]])

Matrix([
[  (b[1]**2 - 4)**(-1/4), (sqrt(b[1]**2 - 4) + b[1])/(2*(b[1]**2 - 4)**(1/4))],
[-1/(b[1]**2 - 4)**(1/4), (sqrt(b[1]**2 - 4) - b[1])/(2*(b[1]**2 - 4)**(1/4))]])

Matrix([
[-sqrt((b[1] - 2)*(b[1] + 2))/2,                             0],
[                             0, sqrt((b[1] - 2)*(b[1] + 2))/2]])

Eq(Matrix([
[utilde(z, mu[1])],
[utilde(z, mu[2])]]), Matrix([
[(sqrt(b[1]**2 - 4) + b[1])*vhat(z, mu[2])/(2*(b[1]**2 - 4)**(1/4)) + uhat(z, mu[1])/(b[1]**2 - 4)**(1/4)],
[(sqrt(b[1]**2 - 4) - b[1])*vhat(z, mu[2])/(2*(b[1]**2 - 4)**(1/4)) - uhat(z, mu[1])/(b[1]**2 - 4)**(1/4)]]))

Eq(Matrix([
[vtilde(z, mu[1])],
[vtilde(z, mu[2])]]), Matrix([
[(sqrt(b[1]**2 - 4) + b[1])*vhat(z, mu[1])/(2*(b[1]**2 - 4)**(1/4)) + uhat(z, mu[2])/(b[1]**2 - 4)**(1/4)],
[(sqrt(b[1]**2 - 4) - b[1])*vhat(z, mu[1])/(2*(b[1]**2 - 4)**(1/4)) - uhat(z, mu[2])/(b[1]**2 - 4)**(1/4)]]))

Eq(Matrix([
[uhat(z, mu[1])],
[vhat(z, mu[2])]]), Matrix([
[(-sqrt(b[1]**2 - 4)/2 - b[1]/2)*utilde(z, mu[2])/(b[1]**2 - 4)**(1/4) + (sqrt(b[1]**2 - 4)/2 - b[1]/2)*utilde(z, mu[1])/(b[1]**2 - 4)**(1/4)],
[                                                               utilde(z, mu[1])/(b[1]**2 - 4)**(1/4) + utilde(z, mu[2])/(b[1]**2 - 4)**(1/4)]]))

Eq(Matrix([
[uhat(z, mu[2])],
[vhat(z, mu[1])]]), Matrix([
[(-sqrt(b[1]**2 - 4)/2 - b[1]/2)*vtilde(z, mu[2])/(b[1]**2 - 4)**(1/4) + (sqrt(b[1]**2 - 4)/2 - b[1]/2)*vtilde(z, mu[1])/(b[1]**2 - 4)**(1/4)],
[                                                               vtilde(z, mu[1])/(b[1]**2 - 4)**(1/4) + vtilde(z, mu[2])/(b[1]**2 - 4)**(1/4)]]))

In [26]:
u_v_tilde_diags = [_.subs(all_hat_as_tilde_subs).doit().subs(sqrt(b[1]**2 -4), x) for _ in du_dv_hat_eqs]
u_v_tilde_diffs = [
    diff(utilde(z, mu[1]),z), diff(utilde(z, mu[2]),z), 
    diff(vtilde(z, mu[1]),z), diff(vtilde(z, mu[2]),z)
]
u_v_tilde_diag_sols = solve(u_v_tilde_diags, u_v_tilde_diffs)
du_dv_tilde_diags = [
    Eq(
        _, 
        u_v_tilde_diag_sols[_]
        .subs(sqrt(b[1]**2 -4), x)
        .expand().collect([utilde(z, mu[1]), utilde(z, mu[2])], factor)
        .subs(x**2, b[1]**2 -4)
        .expand().collect(x, factor)
        .subs(x**2, (b[1]**2 -4).factor())
        .subs(x/((sqrt(b[1]**2 -4))**2).factor(), 1/x)
        .expand().collect(x, factor)
#         .subs(x, sqrt(b[1]**2 -4))
    ) 
    for _ in u_v_tilde_diffs
]
du_dv_tilde_diags_subs = [_.args for _ in du_dv_tilde_diags]

for _ in du_dv_tilde_diags:
    _

Eq(Derivative(utilde(z, mu[1]), z), -(utilde(z, mu[1])**2*vtilde(z, mu[1])*b[1]**2 + 2*utilde(z, mu[1])**2*vtilde(z, mu[2])*b[1]**2 - 4*utilde(z, mu[1])**2*vtilde(z, mu[2]) + 8*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[1]) + 2*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[2])*b[1]**2 + utilde(z, mu[2])**2*vtilde(z, mu[1])*b[1]**2 + 2*utilde(z, mu[2])**2*vtilde(z, mu[2])*b[1]**2 - 4*utilde(z, mu[2])**2*vtilde(z, mu[2]))*b[2]/(2*(b[1] - 2)*(b[1] + 2)) - (-utilde(z, mu[1])**2*vtilde(z, mu[1])*b[1]*b[2] + 2*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[2])*b[1]*b[2] + utilde(z, mu[1])*b[1]**2 - 4*utilde(z, mu[1]) + utilde(z, mu[2])**2*vtilde(z, mu[1])*b[1]*b[2] + 2*utilde(z, mu[2])**2*vtilde(z, mu[2])*b[1]*b[2])/(2*x))

Eq(Derivative(utilde(z, mu[2]), z), (2*utilde(z, mu[1])**2*vtilde(z, mu[1])*b[1]**2 - 4*utilde(z, mu[1])**2*vtilde(z, mu[1]) + utilde(z, mu[1])**2*vtilde(z, mu[2])*b[1]**2 + 2*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[1])*b[1]**2 + 8*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[2]) + 2*utilde(z, mu[2])**2*vtilde(z, mu[1])*b[1]**2 - 4*utilde(z, mu[2])**2*vtilde(z, mu[1]) + utilde(z, mu[2])**2*vtilde(z, mu[2])*b[1]**2)*b[2]/(2*(b[1] - 2)*(b[1] + 2)) + (-2*utilde(z, mu[1])**2*vtilde(z, mu[1])*b[1]*b[2] - utilde(z, mu[1])**2*vtilde(z, mu[2])*b[1]*b[2] - 2*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[1])*b[1]*b[2] + utilde(z, mu[2])**2*vtilde(z, mu[2])*b[1]*b[2] + utilde(z, mu[2])*b[1]**2 - 4*utilde(z, mu[2]))/(2*x))

Eq(Derivative(vtilde(z, mu[1]), z), -(utilde(z, mu[1])*vtilde(z, mu[1])**2*b[1]**2 + 8*utilde(z, mu[1])*vtilde(z, mu[1])*vtilde(z, mu[2]) + utilde(z, mu[1])*vtilde(z, mu[2])**2*b[1]**2 + 2*utilde(z, mu[2])*vtilde(z, mu[1])**2*b[1]**2 - 4*utilde(z, mu[2])*vtilde(z, mu[1])**2 + 2*utilde(z, mu[2])*vtilde(z, mu[1])*vtilde(z, mu[2])*b[1]**2 + 2*utilde(z, mu[2])*vtilde(z, mu[2])**2*b[1]**2 - 4*utilde(z, mu[2])*vtilde(z, mu[2])**2)*b[2]/(2*(b[1] - 2)*(b[1] + 2)) - (-utilde(z, mu[1])*vtilde(z, mu[1])**2*b[1]*b[2] + utilde(z, mu[1])*vtilde(z, mu[2])**2*b[1]*b[2] + 2*utilde(z, mu[2])*vtilde(z, mu[1])*vtilde(z, mu[2])*b[1]*b[2] + 2*utilde(z, mu[2])*vtilde(z, mu[2])**2*b[1]*b[2] + vtilde(z, mu[1])*b[1]**2 - 4*vtilde(z, mu[1]))/(2*x))

Eq(Derivative(vtilde(z, mu[2]), z), (2*utilde(z, mu[1])*vtilde(z, mu[1])**2*b[1]**2 - 4*utilde(z, mu[1])*vtilde(z, mu[1])**2 + 2*utilde(z, mu[1])*vtilde(z, mu[1])*vtilde(z, mu[2])*b[1]**2 + 2*utilde(z, mu[1])*vtilde(z, mu[2])**2*b[1]**2 - 4*utilde(z, mu[1])*vtilde(z, mu[2])**2 + utilde(z, mu[2])*vtilde(z, mu[1])**2*b[1]**2 + 8*utilde(z, mu[2])*vtilde(z, mu[1])*vtilde(z, mu[2]) + utilde(z, mu[2])*vtilde(z, mu[2])**2*b[1]**2)*b[2]/(2*(b[1] - 2)*(b[1] + 2)) + (-2*utilde(z, mu[1])*vtilde(z, mu[1])**2*b[1]*b[2] - 2*utilde(z, mu[1])*vtilde(z, mu[1])*vtilde(z, mu[2])*b[1]*b[2] - utilde(z, mu[2])*vtilde(z, mu[1])**2*b[1]*b[2] + utilde(z, mu[2])*vtilde(z, mu[2])**2*b[1]*b[2] + vtilde(z, mu[2])*b[1]**2 - 4*vtilde(z, mu[2]))/(2*x))

We no longer have wave mixing diagonalisation (i.e., the sum of two products of modes) in the evolution of the modal power, hence this is not a fruitful path for solutions.

In [27]:
duv_1tilde_diag = Eq(
    Derivative(utilde(z, mu[1])*vtilde(z, mu[1]),z),
    Derivative(utilde(z, mu[1])*vtilde(z, mu[1]),z).doit()
    .subs(du_dv_tilde_diags_subs)
    .expand().factor()
)
duv_1tilde_diag

Eq(Derivative(utilde(z, mu[1])*vtilde(z, mu[1]), z), -(2*x*utilde(z, mu[1])**2*vtilde(z, mu[1])**2*b[1]**2*b[2] + 2*x*utilde(z, mu[1])**2*vtilde(z, mu[1])*vtilde(z, mu[2])*b[1]**2*b[2] + 4*x*utilde(z, mu[1])**2*vtilde(z, mu[1])*vtilde(z, mu[2])*b[2] + x*utilde(z, mu[1])**2*vtilde(z, mu[2])**2*b[1]**2*b[2] + 2*x*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[1])**2*b[1]**2*b[2] + 4*x*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[1])**2*b[2] + 4*x*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[1])*vtilde(z, mu[2])*b[1]**2*b[2] + 2*x*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[2])**2*b[1]**2*b[2] - 4*x*utilde(z, mu[1])*utilde(z, mu[2])*vtilde(z, mu[2])**2*b[2] + x*utilde(z, mu[2])**2*vtilde(z, mu[1])**2*b[1]**2*b[2] + 2*x*utilde(z, mu[2])**2*vtilde(z, mu[1])*vtilde(z, mu[2])*b[1]**2*b[2] - 4*x*utilde(z, mu[2])**2*vtilde(z, mu[1])*vtilde(z, mu[2])*b[2] - 2*utilde(z, mu[1])**2*vtilde(z, mu[1])**2*b[1]**3*b[2] + 8*utilde(z, mu[1])**2*vtilde(z, mu[1])**2*b[1]*b[2] + utilde(z, mu[1])**2*vt